# 06.09_All_seurat_Integrate_RPCA_R

Seurat RPCA 整合、BroadType 注释与绘图。

- 当前文件：`analysis/06_single_cell_analysis/06.09_All_seurat_Integrate_RPCA_R.ipynb`
- 原始来源：`Codes/06.08_R_seurat_Integrate_RPCA.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Matrix`, `Seurat`, `dplyr`, `ggplot2`, `patchwork`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

In [ ]:
integrated_rpca

## Seurat RPCA

In [ ]:
library(Seurat)
library(Matrix)
set.seed(42)

In [ ]:
# 1. 路径与物种设置
ogs_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/"
species_order <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
# 定义门类映射关系
phylum_map <- c(
    "Spla"  = "Porifera",
    "ClH23" = "Placozoa", "HoH13" = "Placozoa", "TrH2" = "Placozoa", "TrH1" = "Placozoa",
    "Auco"  = "Cnidaria", "Clhe"  = "Cnidaria", "Neve" = "Cnidaria",
    "Dare"  = "Bilateria"
)

# 2. 数据读取与元数据注入
message(">>> 正在读取 RDS 文件并注入标签...")
objs <- lapply(species_order, function(sp) {
    obj <- readRDS(paste0(ogs_path, sp, ".OG.normalized.rds"))
    
    # [核心修正点] 使用 unname() 去掉 "Dare =" 这种名称干扰
    # 这样赋给 obj$Phylum 的就是一个纯粹的字符串，Seurat 会自动将其广播到所有细胞
    obj$species <- sp
    obj$Phylum  <- unname(phylum_map[sp])
    
    # 设置因子顺序
    obj$Phylum <- factor(obj$Phylum, levels = c("Porifera", "Placozoa", "Cnidaria", "Bilateria"))
    
    return(obj)
})

# 提取并计算共有 OG 集合
ogs_list <- lapply(objs, rownames)
common_ogs <- Reduce(intersect, ogs_list)
message(paste(">>> 共有 OG 数量:", length(common_ogs)))

# 3. 统一子集化 (只保留共有基因)
message(">>> 正在进行基因子集化...")
objs <- lapply(objs, function(obj) {
  return(subset(obj, features = common_ogs))
})

# 4. 数据合并 (Metadata 将会被自动整合)
message(">>> 正在合并对象...")
# add.cell.ids 确保 Barcode 唯一性
merged_raw <- merge(objs[[1]], y = objs[-1], add.cell.ids = species_order)

# 释放原始列表内存
# rm(objs); gc()

# 验证合并后的标签是否准确
message(">>> 合并完成，验证标签分布：")
table(merged_raw$species, merged_raw$Phylum)

In [ ]:
DefaultAssay(objs[[1]])
Assays(objs[[1]])
slotNames(objs[[1]][["RNA"]])

In [ ]:
# ----------------- 1. RPCA 前置处理 (对每个物种独立处理) -----------------
message(">>> 正在为每个物种进行 RPCA 预处理...")

objs_rpca <- lapply(objs, function(x) {
    # 确保当前 Assay 是 RNA
    # DefaultAssay(x) <- "RNA"
    
    # 在 2216 个共有基因中寻找该物种的高变基因
    # 虽然基因总数不多，但这一步对 PCA 的质量至关重要
    # x <- FindVariableFeatures(x, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
    
    # 归一化已经在读入前完成（normalized.rds），直接缩放
    x <- ScaleData(x, features = common_ogs, verbose = FALSE)
    
    # 运行 PCA，作为后续 RPCA 的基础
    # x <- RunPCA(x, features = VariableFeatures(x), npcs = 50, verbose = FALSE)
    x <- RunPCA(x, features = common_ogs, npcs = 50, verbose = FALSE)
    return(x)
})

In [ ]:
# ----------------- 2. 寻找整合锚点 (RPCA 模式) -----------------
message(">>> 正在寻找 RPCA 整合锚点...")

# 提取各物种的高变基因交集作为整合特征
# features <- SelectIntegrationFeatures(object.list = objs_rpca, nfeatures = 2000)

anchors_rpca <- FindIntegrationAnchors(
    object.list = objs_rpca,
    anchor.features = common_ogs,
    # anchor.features = features,
    reduction = "rpca",    # 使用 Reciprocal PCA
    dims = 1:50,           # 通常 30 维足以捕捉物种间共有变异
    # k.anchor = 5           # 如果物种间差异极大，可适当调小此值 (默认5)
)

In [ ]:
# ----------------- 3. 执行数据整合 -----------------
message(">>> 正在执行数据整合 (IntegrateData)...")
integrated_rpca <- IntegrateData(anchorset = anchors_rpca, dims = 1:50)
# 释放中间对象内存
rm(objs_rpca, anchors_rpca); gc()

In [ ]:
table(integrated_rpca$CellTypes)

In [ ]:
# ----------------- 4. 下游分析与可视化 -----------------
message(">>> 正在进行整合后的降维聚类...")

DefaultAssay(integrated_rpca) <- "integrated"

# 整合后的标准流程
integrated_rpca <- ScaleData(integrated_rpca, verbose = FALSE)
integrated_rpca <- RunPCA(integrated_rpca, npcs = 50, verbose = FALSE)
integrated_rpca <- RunUMAP(integrated_rpca, dims = 1:50, verbose = FALSE)
integrated_rpca <- FindNeighbors(integrated_rpca, dims = 1:50, verbose = FALSE)
integrated_rpca <- FindClusters(integrated_rpca, resolution = 0.5)

## 绘制UMAP

In [ ]:
library(ggplot2)

In [ ]:
# ----------------- 5. 注入配色方案与门类标签 -----------------

# 确保配色变量已定义
species_orders <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
species_colors <- c(
    "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
    "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
    "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
)

phylum_orders <- c('Bilateria', 'Cnidaria', 'Placozoa', 'Porifera')
phylum_colors <- c(
    "Porifera"="#fba414", "Placozoa"="#EC2B24", 
    "Cnidaria"="#2A52BE", "Bilateria"="#43b244"
)

In [ ]:
p_rpca_species <- DimPlot(
    integrated_rpca, 
    group.by = "species", 
    cols = species_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        legend.position = "right",      # 图例位置
        legend.direction = "vertical",  # 垂直排列
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14, face = "bold")
    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = species_colors,
        breaks = species_orders
    )
p_rpca_species

ZZ：35绘制UMAP by species

In [ ]:
p_rpca_species <- DimPlot(
    integrated_rpca, 
    group.by = "species", 
    cols = species_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        # 完全去除图例
        legend.position = "none",  # 将 "right" 改为 "none"
        # legend.position = "right",      # 图例位置
        # legend.direction = "vertical",  # 垂直排列
        # legend.text = element_text(size = 12),
        # legend.title = element_text(size = 14, face = "bold"),

         # 去除标题
        plot.title = element_blank(),
        # 去除背景网格
        panel.grid = element_blank()

    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = species_colors,
        breaks = species_orders
    )
p_rpca_species

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_rpca_species, "35.UMAP.9species_by_species", width = 8, height = 8, dpi = 300)

In [ ]:
# 创建临时列，设置因子顺序
integrated_rpca$species_ordered <- factor(
    integrated_rpca$species, 
    levels = species_order  # 指定顺序
)

# 使用临时列绘图
p_rpca_species_split <- DimPlot(
    integrated_rpca, 
    group.by = "species_ordered",  # 使用临时列
    split.by = "species_ordered",  # 使用临时列
    cols = species_colors, 
    ncol = 3,
    raster = TRUE
)

p_rpca_species_split

In [ ]:
p_rpca_phylum <- DimPlot(
    integrated_rpca, 
    group.by = "Phylum", 
    cols = phylum_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        legend.position = "right",      # 图例位置
        legend.direction = "vertical",  # 垂直排列
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14, face = "bold")
    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = phylum_colors,
        breaks = phylum_orders
    )
p_rpca_phylum

In [ ]:
# 创建临时列，设置因子顺序
# integrated_rpca$species_ordered <- factor(
#     integrated_rpca$species, 
#     levels = species_order  # 指定顺序
# )

p_rpca_phylum_split <- DimPlot(
    integrated_rpca, 
    group.by = "Phylum",  # 使用临时列
    split.by = "Phylum",  # 使用临时列
    cols = phylum_colors, 
    ncol = 2,
    raster = TRUE
)

p_rpca_phylum_split

In [ ]:
p_rpca_seurat <- DimPlot(
    integrated_rpca, 
    group.by = "seurat_clusters",
    # pt.size = 0.5, 
    label = TRUE,
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        legend.position = "right",      # 图例位置
        legend.direction = "vertical",  # 垂直排列
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14, face = "bold")
    )
p_rpca_seurat

In [ ]:
# 定义你要提取的物种列表
species_list <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve", "Dare")

# 循环绘图
plot_list <- lapply(species_list, function(sp){
    # 提取该物种的子集
    # 注意：这里提取的是整合后的坐标，但只包含该物种的细胞
    obj_sub <- subset(integrated_rpca, subset = species == sp)
    
    # 绘图
    p <- DimPlot(obj_sub, group.by = "CellTypes", label = TRUE, repel = TRUE, pt.size = 0.2) +
         theme_minimal() +
         theme(panel.grid = element_blank()) +
         NoLegend() + # 因为标签已经 label=TRUE 了，可以关掉侧边图例
         ggtitle(paste("Species:", sp))
    
    return(p)
})

# 如果想在 ipynb 中一次性查看（比如前 4 个）
# library(patchwork)
# wrap_plots(plot_list[1:4], ncol = 2)

# 或者将它们保存到本地文件夹
# for(i in 1:length(plot_list)) {
#   ggsave(paste0("UMAP_", species_list[i], ".pdf"), plot_list[[i]], width = 8, height = 7)
# }

In [ ]:
plot_list[1]
plot_list[2]
plot_list[3]
plot_list[4]
plot_list[5]
plot_list[6]
plot_list[7]
plot_list[8]
plot_list[9]

## 划分细胞类型

In [ ]:
library(Seurat)

# 定义映射列表
mapping <- list(
  # 表皮肌肉类
  "Epidermal/Muscle" = c(
    "Auco_EM, Epidermal/Muscle cell", 
    "Clhe_Epidermal/Muscle", 
    "Dare_Epidermal", 
    "Dare_Endoderm", 
    "Dare_Mesoderm", 
    "HoH13_epithelia", 
    "HoH13_epithelia_gland_like",
    "HoH13_fibre", 
    "Neve_ectoderm.epidermis", 
    "Neve_ectoderm.embryonic", 
    "Neve_ectoderm.pharyngeal",
    "Neve_retractor muscle", 
    "Spla_Endymocytes", 
    "TrH1_epithelia", 
    "TrH1_epithelia_gland_like", 
    "TrH1_fibre", 
    "TrH2_epithelia", 
    "TrH2_epithelia_gland_like",
    "TrH2_fibre", 
    "ClH23_epithelia", 
    "ClH23_epithelia_gland_like",
    "ClH23_fibre"
  ),
  # 消化类
  # "Digestive" = c(
  #   "Auco_GA, Gastrodermal cell", 
  #   "Clhe_Gastroderm", 
  #   "Neve_gastrodermis"
  # ),          
  # 神经类
  "Neural" = c(
    "Auco_NE, Neural cell", 
    "Clhe_Neural", 
    "Dare_Neural Anterior", 
    "Dare_Neural Mid", 
    "Dare_Neural Posterior", 
    "Dare_Neural Crest", 
    "HoH13_peptidergic", 
    "Neve_neuronal", 
    "Neve_NPC", 
    "Spla_Amoeboid-Neuroid", 
    "TrH1_peptidergic", 
    "TrH2_peptidergic", 
    "ClH23_peptidergic"
  ),
  # 感觉类
  "Sensory" = c(
    "Auco_HA, Hair cell", 
    "Clhe_Bioluminescent Cells"
  ),
  # 腺体类
  "Gland" = c(
    # 消化
    "Auco_GA, Gastrodermal cell", 
    "Clhe_Gastroderm", 
    "Neve_gastrodermis",
    # 腺体
    "Auco_GL, Gland cell", 
    "Clhe_Gland Cell", 
    "HoH13_gland", 
    "HoH13_lipophil", 
    "Neve_gland.mucous", 
    "Neve_secretory", 
    "Spla_Peptidocytes", 
    "TrH1_gland", 
    "TrH1_lipophil", 
    "TrH2_gland", 
    "TrH2_lipophil", 
    "ClH23_gland", 
    "ClH23_lipophil"
  ),
  # 干细胞
  "Stem/Germline" = c(
    "Auco_SG, Stem/Germline cell", 
    "Clhe_Stem Cell/Germ Cell", 
    "Dare_Germline", 
    "Neve_mesendoderm.embryonic",
    "Spla_Archeocytes and relatives", 
    "TrH1_meiotic", 
    "TrH2_meiotic"
  ),  
  # 刺细胞
  "Cnidocytes" = c(
    "Auco_CN, Cnidocytes/Nematocyte cell", 
    "Clhe_Nematocyte", 
    "Neve_cnidocyte", 
    "Neve_cnidocyte.mature"
  ),
  # 未知
  "Unknow" = c(
    "HoH13_trans", 
    "HoH13_unknown_1", 
    "Spla_transitional", 
    "TrH1_trans", 
    "TrH2_trans", 
    "TrH2_unknown_1", 
    "ClH23_trans", 
    "ClH23_unknown_1"
  )
)

# 构建反向查找向量
broad_map <- unlist(lapply(names(mapping), function(n) setNames(rep(n, length(mapping[[n]])), mapping[[n]])))

# 注入 metadata
# 执行映射并去掉名字
broad_vec <- unname(broad_map[integrated_rpca$CellTypes])
# 检查长度是否完全一致 (必须等于细胞总数 173786)
if (length(broad_vec) == ncol(integrated_rpca)) {
    integrated_rpca$BroadType <- broad_vec
    message(">>> BroadType 注入成功！")
} else {
    stop("长度不匹配，请检查映射逻辑。")
}
# 检查是否有因为拼写不一致导致的 NA
if (any(is.na(integrated_rpca$BroadType))) {
    message("警告：检测到 NA 值，以下 CellTypes 未能成功映射：")
    print(unique(integrated_rpca$CellTypes[is.na(integrated_rpca$BroadType)]))
}

# 检查是否有漏掉的分类 (如果输出 0 则表示全部归类成功)
na_count <- sum(is.na(integrated_rpca$BroadType))
if(na_count > 0) {
  warning(paste("发现", na_count, "个细胞未匹配到大类，请检查 CellTypes 拼写是否完全一致！"))
  print(unique(integrated_rpca$CellTypes[is.na(integrated_rpca$BroadType)]))
} else {
  message(">>> 所有细胞类型已成功归类。")
}

In [ ]:
BroadType_orders <- c(
    "Cnidocytes", 
    "Epidermal/Muscle", 
    # "Digestive", 
    "Gland", 
    "Sensory", 
    "Neural", 
    "Stem/Germline", 
    "Unknow"
)

BroadType_colors <- c(
    'Cnidocytes'= '#17becf',         # CN 青色 '#17becf'
    'Epidermal/Muscle'= '#9467bd',   # EM 紫色 '#9467bd'
    # 'Digestive'= '#ff7f0e',          # GA 橙色 '#ff7f0e'
    'Gland'= '#d62728',              # GL 红色 '#d62728'
    'Sensory'= '#8c564b',            # HA 棕色 '#8c564b'
    'Neural'= '#2ca02c',             # NE 绿色 '#2ca02c'
    'Stem/Germline'= '#fedb61',      # SG 黄色 '#fedb61'
    'Unknow'= '#707070'
)

p_rpca_BroadType <- DimPlot(
    integrated_rpca, 
    group.by = "BroadType", 
    cols = BroadType_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        legend.position = "right",      # 图例位置
        legend.direction = "vertical",  # 垂直排列
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14, face = "bold")
    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = BroadType_colors,
        breaks = BroadType_orders
    )
p_rpca_BroadType

ZZ：36绘制UMAP by BroadType

In [ ]:
p_rpca_BroadType <- DimPlot(
    integrated_rpca, 
    group.by = "BroadType", 
    cols = BroadType_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        legend.position = "none",

        # 去除标题
        plot.title = element_blank(),
        # 去除背景网格
        panel.grid = element_blank()
    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = BroadType_colors,
        breaks = BroadType_orders
    )
p_rpca_BroadType

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_rpca_BroadType, "36.UMAP.9species_by_BroadType", width = 8, height = 8, dpi = 300)

In [ ]:
# ---------- 8) 保存结果 ----------
# 可选：导出嵌入与元数据（便于后续评估/作图）
dir.create("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA", showWarnings = FALSE, recursive = TRUE)
# 导出RDS
saveRDS(integrated_rpca, file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/integrated_rpca.rds")
# 导出pca
write.csv(Embeddings(integrated_rpca, "pca"),
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/pca_embeddings.csv")
# 导出umap
write.csv(Embeddings(integrated_rpca, "umap"),
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/umap_embeddings.csv")
# 导出metadata
write.csv(integrated_rpca@meta.data,
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/metadata.csv")

In [ ]:
# 提取 integrated assay 中的 feature 名字
features <- rownames(integrated_rpca)

# 导出为文本文件 (每行一个 OG)
write.table(features, file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/integrated_features_list.txt", 
            row.names = FALSE, col.names = FALSE, quote = FALSE)

message(">>> 已导出 ", length(features), " 个 Feature 编号。")

## 统计每个BroadType中species的数量分布

In [ ]:
library(Seurat)
library(Matrix)
library(ggplot2)

In [ ]:
integrated_rpca <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/integrated_rpca.rds")

In [ ]:
# 确保配色变量已定义
species_orders <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
species_colors <- c(
    "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
    "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
    "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
)

phylum_orders <- c('Bilateria', 'Cnidaria', 'Placozoa', 'Porifera')
phylum_colors <- c(
    "Porifera"="#fba414", "Placozoa"="#EC2B24", 
    "Cnidaria"="#2A52BE", "Bilateria"="#43b244"
)

BroadType_orders <- c(
    "Cnidocytes", 
    "Epidermal/Muscle", 
    # "Digestive", 
    "Gland", 
    "Sensory", 
    "Neural", 
    "Stem/Germline", 
    "Unknow"
)

BroadType_colors <- c(
    'Cnidocytes'= '#17becf',         # CN 青色 '#17becf'
    'Epidermal/Muscle'= '#9467bd',   # EM 紫色 '#9467bd'
    # 'Digestive'= '#ff7f0e',          # GA 橙色 '#ff7f0e'
    'Gland'= '#d62728',              # GL 红色 '#d62728'
    'Sensory'= '#8c564b',            # HA 棕色 '#8c564b'
    'Neural'= '#2ca02c',             # NE 绿色 '#2ca02c'
    'Stem/Germline'= '#fedb61',      # SG 黄色 '#fedb61'
    'Unknow'= '#707070'
)

In [ ]:
# 生成频数表
BroadType_species_table <- table(integrated_rpca$BroadType, integrated_rpca$species)

# 转换为 Data Frame 方便后续处理
BroadType_species_df <- as.data.frame(BroadType_species_table)
colnames(BroadType_species_df) <- c("BroadType", "Species", "CellCount")

# 查看前几行
head(BroadType_species_df)

In [ ]:
write.csv(BroadType_species_df,
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/BroadType_species_df.csv")

In [ ]:
# 读取文件
BroadType_species_df <- read.csv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/BroadType_species_df.csv")

In [ ]:
# 定义之前确定的物种配色
# species_colors <- c(
#     "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
#     "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
#     "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
# )

# 确保 Species 是 Factor 且按照你指定的顺序排列
# 假设你的物种顺序是 species_order <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
species_order <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
BroadType_species_df$Species <- factor(BroadType_species_df$Species, levels = species_order)

# 绘图，绘制百分比堆叠图
ggplot(BroadType_species_df, aes(x = BroadType, y = CellCount, fill = Species)) +
  geom_bar(stat = "identity", position = "fill") + 
  scale_fill_manual(values = species_colors) +
  theme_minimal() +
  labs(y = "Proportion of Cells", 
       title = "Species Composition per BroadType",
       x = "BroadType") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    # 保持面板整洁
    panel.grid.major.x = element_blank()
  )

ZZ：37绘制细胞类型占比堆叠图

In [ ]:
p_species_composition <- ggplot(BroadType_species_df, aes(x = BroadType, y = CellCount, fill = Species)) +
  geom_bar(stat = "identity", position = "fill") + 
  scale_fill_manual(values = species_colors) +
  theme_minimal() +
  labs(y = "Proportion of Cells", 
       title = "Species Composition per BroadType",
       x = "Functional Group (BroadType)") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    # 保持面板整洁
    panel.grid.major.x = element_blank()
  )
p_species_composition

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_species_composition, "37.Bar.9species_composition_per_BroadType", width = 8, height = 6, dpi = 300)

In [ ]:
# 查询OG
library(dplyr)


markers_list <- c(
  # 刺细胞
  'XLOC-010665#MYC-MARMO#P22555',
  
  # 表皮肌肉细胞
  # 'XLOC-011330#CALM-MACPY#Q40302',
  'XLOC-004173#DD3-DICDI#Q58A42',
  'XLOC-026552#MLE-BRAFL#Q17133',
  'XLOC-014937#SVIL-BOVIN#O46385',
  'XLOC-016636#MYL6B-HUMAN#P14649',
  # 'XLOC-018113#MYL6B-HUMAN#P14649',
  # 'XLOC-026007#TPM1-PODCA#P41114',
  # 'XLOC-024876#TPM2-PODCA#Q9U5M4',
  # 'gene-evm.model.ptg000006l.268#MYL1-DANRE#Q6P0G6',
  # 'XLOC-016635#MYL1-DANRE#Q6P0G6',
  # 'gene-evm.model.ptg000002l.132#MLE-BRAFL#Q17133'

  # 胃皮层细胞
  # 'XLOC-003006#APLP-LOCMI#Q9U943',
  # 'XLOC-006269#CATL1-DROME#Q95029',
  'XLOC-009813#IRF2-CHICK#Q98925',
  'XLOC-002530#CSMD1-MOUSE#Q923L3',

  # 腺体细胞
  'XLOC-002769#CTRB2-HUMAN#Q6GPI1',
  'gene-evm.model.ptg000015l.30#CTR2-CANLF#P04813',
  'XLOC-020391#TRY1-HUMAN#P07477',
  # 'XLOC-001150#SVEP1-RAT#P0C6B8',
  'XLOC-000359#SVEP1-MOUSE#A2AVA0',
  'XLOC-006327#CEL2A-RAT#P00774',

  # 感觉细胞
  'XLOC-001822#PKD2-BOVIN#Q4GZT3',
  'XLOC-015184#CAC1E-MOUSE#Q61290',
  'gene-evm.model.ptg000003l.332#TBA1A-CHICK#P02552',
  'XLOC-006274#AVIL-RAT#Q9WU06',

  # 神经细胞
  'XLOC-019965#SYT14-HUMAN#Q8NB59',
  'gene-evm.model.ptg000013l.805#CAB32-DROME#P41044',
  'gene-evm.model.ptg000022l.585#ACH1-CAEEL#P48180',
  'gene-evm.model.ptg000001l.686#SOX14-DANRE#Q32PP9',

  # 干细胞/生殖细胞
  # 'gene-evm.model.ptg000031l.94#SMC2-XENLA#P50533',
  'XLOC-003196#SMC2-XENLA#P50533',
  'XLOC-022183#SMC4-XENLA#P50532'
)

# 1. 读取 protein 与 OG 的对应关系表
# 假设文件在当前工作目录
auco_og_map <- read.csv("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/OrthoFinder_Inference/OrthoFinder_output/Step4_SpeciesOrthogroup/Auco.protein_to_orthogroup.csv")

# 2. 提取这些 Marker 对应的 OGs
# 注意：确保 csv 中的 protein_id 与 markers_list 的格式完全一致（是否有前缀/后缀）
marker_to_og <- auco_og_map %>%
  filter(protein_id %in% markers_list) %>%
  select(protein_id, orthogroup)

# 3. 查看查询结果
print(marker_to_og)

# 4. 提取唯一的 OG 编号用于下一步分析
target_ogs <- unique(marker_to_og$orthogroup)
print(target_ogs)

In [ ]:
# 1. 获取整合对象中实际存在的 Features (OGs)
integrated_features <- rownames(integrated_rpca)

# 2. 与 target_ogs 取交集
# intersect 函数会自动去重并保留两个向量中都存在的元素
marker_OGs <- intersect(target_ogs, integrated_features)

# 3. 打印统计信息，检查有多少基因成功“入选”
cat(">>> 原始查询的 target_ogs 数量:", length(target_ogs), "\n")
cat(">>> 在整合数据中实际存在的 OGs 数量:", length(marker_OGs), "\n")

marker_OGs

In [ ]:
library(Seurat)
library(ggplot2)

markers_list <- c(
  "OG0000203",
  "OG0000133",
  "OG0000036",
  "OG0000260",
  "OG0000112",
  "OG0000166"

  # "OG0000036",
  # "OG0000112",
  # "OG0000166",
  # "OG0002183",
  # "OG0000203",
  # "OG0000260",
  # "OG0003240",
  # "OG0000133",
  # "OG0000247",
  # "OG0000225",
  # "OG0000237",
  # "OG0004822"


  # # 表皮肌肉
  # 'OG0004331',
  # 'OG0006660',
  # # 腺体细胞
  # 'OG0000015',
  # # 胃皮层
  # 'OG0000429',
  # # 感觉细胞
  # 'OG0000612',
  # # 神经细胞
  # 'OG0000166',
  # 'OG0000198',
  # 'OG0001111'
)

# 1. 确保 BroadType 的因子顺序（控制大类在组内的排列）
broad_type_order <- c("Cnidocytes", "Epidermal/Muscle", "Gland", 
                      "Sensory", "Neural", "Stem/Germline", "Unknow")

integrated_rpca$BroadType <- factor(as.character(integrated_rpca$BroadType), 
                                    levels = broad_type_order)

# 2. 确保 species 的因子顺序（控制物种在组间的排列）
# 建议按照进化树顺序排列：Spla -> TrH1 -> Auco -> ... -> Dare
species_order <- c("Spla", "TrH1", "TrH2", "HoH13", "ClH23", "Auco", "Clhe", "Neve", "Dare")
integrated_rpca$species <- factor(integrated_rpca$species, levels = species_order)


p <- DotPlot(
  integrated_rpca,
  features = markers_list,
  group.by = "BroadType",
  split.by = "species",      # <--- 核心修改：按物种拆分
  cols = "RdYlBu"            # 如果是 split.by，指定颜色方案（或保持默认）
) +
  coord_flip() +  # <--- 核心修改：横纵坐标互换
  # theme_classic() +
  scale_color_viridis_c() +
  theme(
      panel.grid.major = element_line(colour = "grey95"),
      axis.title = element_text(size = 14, face = "bold"),
      axis.text.y = element_text(size = 10, face = "italic"), # 基因名斜体
      axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 8), # X轴文字较多，建议旋转90度
      legend.position = "right",
      strip.background = element_blank() # 如果有 strip 标签可以去掉背景
    ) +
    labs(x = "Orthogroups (Markers)", y = "Species-specific BroadTypes")

p

ZZ：38绘制Dotplot图

In [ ]:
p_dotplot_markerOGs <- DotPlot(
  integrated_rpca,
  features = markers_list,
  group.by = "BroadType",
  split.by = "species",      # <--- 核心修改：按物种拆分
  cols = "RdYlBu"            # 如果是 split.by，指定颜色方案（或保持默认）
) +
  coord_flip() +  # <--- 核心修改：横纵坐标互换
  # theme_classic() +
  scale_color_viridis_c() +
  theme(
      panel.grid.major = element_line(colour = "grey95"),
      axis.title = element_text(size = 14, face = "bold"),
      axis.text.y = element_text(size = 10, face = "italic"), # 基因名斜体
      axis.text.x = element_text(angle = 30, vjust = 0.5, hjust = 1, size = 8), # X轴文字较多，建议旋转90度
      legend.position = "right",
      strip.background = element_blank() # 如果有 strip 标签可以去掉背景
    ) +
    labs(x = "Orthogroups (Markers)", y = "Species-specific BroadTypes")

p_dotplot_markerOGs
# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_dotplot_markerOGs, "38.DotPlot.9species_BroadType_markerOGs", width = 15, height = 6, dpi = 300)

ZZ：39绘制整合后的UMAP,只展示每个门类的神经相关的细胞类型

In [ ]:
# 1. 提取 Porifera 门类的子集
porifera_obj <- subset(integrated_rpca, subset = Phylum == "Porifera")

# 2. 确定要高亮的 Neural 细胞 ID
neural_cells <- WhichCells(porifera_obj, expression = BroadType == "Neural")

# 3. 绘图：使用 cells.highlight 实现黑色高亮
# 这种方法会自动将非高亮细胞置灰，我们可以通过 cols 指定背景色
p_porifera <- DimPlot(
    porifera_obj, 
    cells.highlight = neural_cells,
    cols.highlight = "#815c94",   # Neural 点设为黑色
    sizes.highlight = 0.5,      # 适当加大高亮点的尺寸
    cols = "#fba414",            # 其余点自定义为浅灰色（或换成您的特定色号，如 "#E5E5E5"）
    raster = FALSE              # 子集细胞数通常不多，关闭 raster 可以获得更清晰的矢量图
) +
  theme_void() +               # 去除背景坐标轴，适合展示演化拓扑
  theme(
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    legend.position = "none"
  )

print(p_porifera)

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_porifera, "39.UMAP.Porifera_Neural", width = 8, height = 8, dpi = 300)

In [ ]:
placozoa_obj <- subset(integrated_rpca, subset = Phylum == "Placozoa")

# 2. 确定要高亮的 Neural 细胞 ID
neural_cells <- WhichCells(placozoa_obj, expression = BroadType == "Neural")

# 3. 绘图：使用 cells.highlight 实现黑色高亮
# 这种方法会自动将非高亮细胞置灰，我们可以通过 cols 指定背景色
p_placozoa <- DimPlot(
    placozoa_obj, 
    cells.highlight = neural_cells,
    cols.highlight = "#815c94",   # Neural 点设为黑色
    sizes.highlight = 0.05,      # 适当加大高亮点的尺寸
    cols = "#EC2B24",            # 其余点自定义为浅灰色（或换成您的特定色号，如 "#E5E5E5"）
    raster = FALSE              # 子集细胞数通常不多，关闭 raster 可以获得更清晰的矢量图
) +
  theme_void() +               # 去除背景坐标轴，适合展示演化拓扑
  theme(
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    legend.position = "none"
  )

print(p_placozoa)

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_placozoa, "40.UMAP.Placozoa_Neural", width = 8, height = 8, dpi = 300)

In [ ]:
cnidaria_obj <- subset(integrated_rpca, subset = Phylum == "Cnidaria")

# 2. 确定要高亮的 Neural 细胞 ID
neural_cells <- WhichCells(cnidaria_obj, expression = BroadType == "Neural")

# 3. 绘图：使用 cells.highlight 实现黑色高亮
# 这种方法会自动将非高亮细胞置灰，我们可以通过 cols 指定背景色
p_cnidaria <- DimPlot(
    cnidaria_obj, 
    cells.highlight = neural_cells,
    cols.highlight = "#815c94",   # Neural 点设为黑色
    sizes.highlight = 0.05,      # 适当加大高亮点的尺寸
    cols = "#2A52BE",            # 其余点自定义为浅灰色（或换成您的特定色号，如 "#E5E5E5"）
    raster = FALSE              # 子集细胞数通常不多，关闭 raster 可以获得更清晰的矢量图
) +
  theme_void() +               # 去除背景坐标轴，适合展示演化拓扑
  theme(
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    legend.position = "none"
  )

print(p_cnidaria)

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_cnidaria, "41.UMAP.Cnidaria_Neural", width = 8, height = 8, dpi = 300)

In [ ]:
bilateria_obj <- subset(integrated_rpca, subset = Phylum == "Bilateria")

# 2. 确定要高亮的 Neural 细胞 ID
neural_cells <- WhichCells(bilateria_obj, expression = BroadType == "Neural")

# 3. 绘图：使用 cells.highlight 实现黑色高亮
# 这种方法会自动将非高亮细胞置灰，我们可以通过 cols 指定背景色
p_bilateria <- DimPlot(
    bilateria_obj, 
    cells.highlight = neural_cells,
    cols.highlight = "#815c94",   # Neural 点设为黑色
    sizes.highlight = 0.05,      # 适当加大高亮点的尺寸
    cols = "#43b244",            # 其余点自定义为浅灰色（或换成您的特定色号，如 "#E5E5E5"）
    raster = FALSE              # 子集细胞数通常不多，关闭 raster 可以获得更清晰的矢量图
) +
  theme_void() +               # 去除背景坐标轴，适合展示演化拓扑
  theme(
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    legend.position = "none"
  )

print(p_bilateria)

# ============= 保存图片 =============
# 定义保存函数
save_plot <- function(plot, filename, width = 8, height = 6, dpi = 300) {
    # 保存为PDF
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".pdf")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "pdf"
    )
    
    # 保存为PNG
    ggsave(
        filename = file.path(fig_dir, paste0(filename, ".png")),
        plot = plot,
        width = width,
        height = height,
        dpi = dpi,
        units = "in",
        bg = "white",
        device = "png"
    )
    
    cat("图片已保存至", fig_dir, "目录：\n")
    cat("-", paste0(filename, ".pdf\n"))
    cat("-", paste0(filename, ".png\n"))
}

# 保存图片
save_plot(p_bilateria, "42.UMAP.Bilateria_Neural", width = 8, height = 8, dpi = 300)